# Layered Model for Wavefield output

model has earth-packed snow (older snow)-fresh snow-air

rest of the code from wavefield_output_layered

In [ ]:
%matplotlib inline

In [ ]:
# Imports (taken from salvus tutorials)
import os

SALVUS_FLOW_SITE_NAME = os.environ.get("salome_remote") # Site name given in the installation of Salvus flow
PROJECT_DIR = "simulation_wavefield_output_multi"  
import pathlib
import numpy as np
import salvus.namespace as sn
import xarray as xr
import salvus.namespace as sn
from salvus.project.tools.processing import block_processing
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import matplotlib.pyplot as plt
from matplotlib import animation
import obspy

# For wavefield output code
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers import wavefield_output

# Keep .gitignore updated so Salvus project output is not tracked.
from pathlib import Path

gitignore_path = Path(".gitignore")
entry = f"{PROJECT_DIR}/"

if gitignore_path.exists():
    lines = gitignore_path.read_text(encoding="utf-8").splitlines()
else:
    lines = []

if entry not in lines:
    with gitignore_path.open("a", encoding="utf-8") as f:
        if lines and lines[-1] != "":
            f.write("\n")
        f.write(entry + "\n")
    print(f"Added to .gitignore: {entry}")
else:
    print(f"Already in .gitignore: {entry}")


Already in .gitignore: simulation_wavefield_output_multi/


In [ ]:
# # Delete previous events if re-running the notebook
# p.events.delete(event_name="event_wavefield_output")

# # Also delete previous model configurations if re-running the notebook
# path = pathlib.Path(PROJECT_DIR) / "INTERNAL" / "ENTITIES" / "SIMULATION_CONFIGURATIONS"
# for file in path.glob("sim_2d_layered_multi.json"):
#     file.unlink()

In [ ]:
# Setup of the model domain as a box 
domain_2d = sn.domain.dim2.BoxDomain(x0=0, x1=133, y0=0, y1=4)
p = sn.Project.from_domain(path=PROJECT_DIR, domain=domain_2d, load_if_exists=True)

Accordion(children=(HTML(value='\n                <head>\n                <style>\n                td {\n     …

[2026-03-24 09:44:25,319] INFO: Loading project from simulation_wavefield_output_multi.


In [ ]:
# # Delete entire project directory to clear all cached metadata
# import shutil
# if pathlib.Path(PROJECT_DIR).exists():
#     shutil.rmtree(PROJECT_DIR)
#     print(f"Deleted {PROJECT_DIR}")

In [ ]:



# Layered model setup according to mondaic docs
# Minimal and maximal x extent: same as domain box
x_min = 0.0
x_max = 133.0

# Defining extent of löayers (layers_x) and thickness / topography of layers (layers_y)
layers_x = [
    np.array([0.0, 133.0]),  # top boundary
    np.array([0.0, 133.0]),  # snow-air interface
    np.array([0.0, 133.0]),  # compacted snow - fresh snow interface
    np.array([0.0, 133.0]),  # earth-snow interface 
    np.array([0.0, 133]),  # bottom boundary
]

layers_y = [
    np.array([4.0, 4.0]),
    np.array([3.0, 3.0]),        
    np.array([2.0, 2.0]),        
    np.array([1.0, 1.0]),        
    np.array([0.0, 0.0]),        
]


# Defining model parameters (vp, vs and rho) for earth, snow and air, earth and air velocities taken from https://pburnley.faculty.unlv.edu/GEOL452_652/seismology/notes/SeismicNotes10RVel.html
vp = np.array([2200, 300, 150, 332])
vs = np.array([880, 150, 75, 0])
rho = np.array([2000, 180, 120, 1.2250])


interpolation_styles = ["linear"] * len(layers_x)


splines = sn.toolbox.get_interpolating_splines(
    layers_x, layers_y, kind=interpolation_styles
)

# # Plotting the layer boundaries to check if they are correct
# f = plt.figure(figsize=(10, 5))
# x_plot = np.linspace(x_min, x_max)
# for top, bot in splines:
#     plt.plot(x_plot, top(x_plot))
#     plt.plot(x_plot, bot(x_plot))

# plt.xlabel("x (m)")
# plt.ylabel("y (m)")
# plt.title("Interfaces")
# plt.ylim(0,1.5)

# Genetarte mesh
# Maximum frequency to resolve with elements_per_wavelength.
max_frequency = 50

# Print lenght of splines because of size mismatch between splines and vs
shp = len(splines)
print(shp)

slowest_velocities = np.array([
    880,   # earth
    150, 
    75,  # snow
    75,   # air layer meshing controlled by snow below --> need this because else slowest_velocities gives an errror because it goes to infinity
])

# Generate the mesh
mesh, bnd = sn.toolbox.generate_mesh_from_splines_2d(
    x_min=0,
    x_max=x_max,
    splines=splines,
    elements_per_wavelength=2,
    maximum_frequency=max_frequency,
    use_refinements=True,
    slowest_velocities=slowest_velocities,
    absorbing_boundaries=(["x0", "x1", "y0"], 10.0), # Change this if different boundaries need to be absorbing 
)

mesh = np.sum(mesh)

# Add info about absorbing boundaries CHANGE DEPENDING ON WHICH BOUNDARIES NEED TO BE TRANSPARENT / ABSORBING
mesh.attach_global_variable("max_dist_ABC", bnd)
mesh.attach_global_variable("ABC_side_sets", ", ".join(["x0", "x1", "y0"]))
mesh.attach_global_variable("ABC_vel", float(min(vs[vs > 0])))
mesh.attach_global_variable("ABC_freq", max_frequency / 2.0)
mesh.attach_global_variable("ABC_nwave", 5.0)


# Attaching parameters (vp,vs,rho) to mesh 
nodes = mesh.get_element_nodes()[:, :, 0]
vp_a, vs_a, ro_a = np.ones((3, *nodes.shape))
for _i, (vp_val, vs_val, ro_val) in enumerate(zip(vp, vs, rho)):
    # Find which elements are in a given region.
    idx = np.where(mesh.elemental_fields["region"] == _i)

    # Set parameters in that region to a constant value.
    vp_a[idx] = vp_val
    vs_a[idx] = vs_val
    ro_a[idx] = ro_val

# Attach parameters.
for k, v in zip(["VP", "VS", "RHO"], [vp_a, vs_a, ro_a]):
    mesh.attach_field(k, v)

# Attach acoustic / elastic flag.
mesh_2d_layered_multi = sn.toolbox.detect_fluid(mesh)

# # Checking which values are assigned to which layer: LAYER 0 IS THE BOTTOM LAYER
# np.unique(mesh.elemental_fields["region"])
# for i in range(3):
#     idx = mesh.elemental_fields["region"] == i
#     print(i,
#           np.unique(mesh.elemental_fields["VP"][idx]),
#           np.unique(mesh.elemental_fields["VS"][idx]),
#           np.unique(mesh.elemental_fields["RHO"][idx]))


# # Plot Mesh toc heck
# mesh_2d_layered_multi



Accordion()

4


In [ ]:
# Soure located at the top of the domain 
src = sn.simple_config.source.cartesian.VectorPoint2D(
    x=66.5,
    y=2.5,
    fx=0.0,
    fy=-1.0,
) # fx and fy values dependend on the type and force of source

p.add_to_project(sn.Event(event_name="event_wavefield_output", sources=[src]))

In [ ]:
# For layered model 
p.add_to_project(
    sn.UnstructuredMeshSimulationConfiguration(
        name="sim_2d_layered",
        unstructured_mesh=mesh_2d_layered_multi,
        event_configuration=sn.EventConfiguration(
        wavelet=sn.simple_config.stf.Ricker(center_frequency=10,
        time_shift_in_seconds=0.3), 
        waveform_simulation_configuration=sn.WaveformSimulationConfiguration(
            start_time_in_seconds=-0.3,
            end_time_in_seconds=2.0,
            spectral_element_order=2
            ),
        ),
    ),
)
# p.viz.nb.simulation_setup("sim_2d_layered", events=["event_wavefield_output"])

In [ ]:
# Layered
p.simulations.launch(
    simulation_configuration="sim_2d_layered",
    events=p.events.list(),
    site_name="salome_remote", 
    ranks_per_job=8,
    extra_output_configuration={
        "volume_data": {
            "sampling_interval_in_time_steps": 10,
            "fields": ["velocity", "displacement"],
        },
    },
)
p.simulations.query(block=True)


[2026-03-23 22:51:00,080] INFO: Submitting job ...
Uploading 1 files...

🚀  Submitted job_2603232251161367_00c4f1692b@salome_remote


VBox()

In [ ]:
# Layered model 
out_2d_layered_multi = p.simulations.get_simulation_output_directory("sim_2d_layered", "event_wavefield_output")

output_dir = pathlib.Path(out_2d_layered_multi)
h5_file = output_dir / "volume_data_output.h5"

# Fallback: find the actual HDF5 file if the default name/path is missing.
if not h5_file.exists():
    h5_candidates = sorted(output_dir.rglob("*.h5"))
    volume_candidates = [f for f in h5_candidates if "volume" in f.name.lower()]
    if volume_candidates:
        h5_file = volume_candidates[0]
    else:
        raise FileNotFoundError(
            f"No volume HDF5 output found in: {output_dir}\n"
            f"Available files: {[f.name for f in output_dir.rglob('*') if f.is_file()]}"
        )

vel_wo_layered = wavefield_output.WavefieldOutput.from_file(
    h5_file,
    "velocity",
    "volume",
)

# Converting to an x array
vel_2d_layered_multi = wavefield_output.wavefield_output_to_xarray(
    vel_wo_layered,
    points=[np.linspace(0, 1, 101), np.linspace(0, 1, 101)],
)

# Plotting wavefield output
# T is the timeslice (the t'th output image)
vel_2d_layered_multi.isel(c=1, t=35).T.plot(shading="gouraud", infer_intervals=False)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'simulation_wavefield_output_multi/EVENTS/event_wavefield_output/WAVEFORM_DATA/INTERNAL/a8/cd/3cbd1df77596/volume_data_output.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["figure.dpi"] = 150
plt.ioff()

# getting wavefield output like above
out_2d_layered = p.simulations.get_simulation_output_directory(
    "sim_2d_layered", "event_wavefield_output"
)

vel_wo_layered = wavefield_output.WavefieldOutput.from_file(
    pathlib.Path(out_2d_layered, "volume_data_output.h5"),
    "velocity",
    "volume",
)

vel_2d_layered = wavefield_output.wavefield_output_to_xarray(
    vel_wo_layered,
    points=[np.linspace(0, 1, 101), np.linspace(0, 1, 101)],
)

# Setting up animation
fig, ax = plt.subplots()


def animate(t):
    for coll in ax.collections:
        coll.remove()

    vel_2d_layered.isel(c=1, t=t).T.plot(
        ax=ax,
        shading="gouraud",
        infer_intervals=False,
        add_colorbar=False,
    )

    # Force axis limits

    # ax.set_ylim(0.35, 1.0)

    # Kill autoscaling 
    ax.set_autoscale_on(False)

    ax.set_title(f"Velocity field (t = {t})")


    # Creating animation
ani_layered_multi = animation.FuncAnimation(
    fig,
    animate,
    frames=vel_2d_layered.sizes["t"],
    interval=100 # chnage interval to a higherr value for slower animation
)

ani_layered_multi



In [ ]:

# Define receiver line at snow-earth interface
#y_surface = 3 * 2 / 3  # middle of snow layer 
y_surface = 3 * 1 / 3 # snow-air interface
x_line = np.linspace(0.0, 300.0, 1001)
y_line = np.full_like(x_line, y_surface)

# Extract wavefield along receiver line
vel_sg = wavefield_output.wavefield_output_to_xarray(
    vel_wo_layered,
    points=np.column_stack((x_line, y_line)),
)

# Select vertical component and full time range
sg_vy = vel_sg.isel(c=1)

t_vals = sg_vy.t.values
data = sg_vy.values  # shape: (n_t, n_points)

print("t range:", t_vals[0], "->", t_vals[-1])
print("data shape:", data.shape)

# Clip colorscale (robust equivalent)
vmax = np.percentile(np.abs(data), 95)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.pcolormesh(
    x_line, t_vals, data,
    shading="gouraud",
    cmap="RdBu_r",
    vmin=-vmax, vmax=vmax,
)
ax.invert_yaxis()
ax.axvline(x=150, color="teal", lw=0.8, linestyle="--", label="source x=150")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Time (s)")
ax.set_xlim(0, 300)
ax.set_title("Seismic waterfall plot - vertical particle velocity at halfway depth in snow (middle of snow layer)")
#ax.set_title("Seismic waterfall plot - vertical particle velocity at snow-earth interface")
ax.legend()
plt.colorbar(im, ax=ax, label="Vertical particle velocity (m/s)")
plt.tight_layout()
plt.show()

In [ ]:

# Extracting trace, picking trace far away from source so that source doesnt dominate
trace = vel_sg.isel(c=1).sel(point=0.0, method="nearest").sel(t=slice(0, 2.0))
t_vals = trace.t.values
y_vals = trace.values

# dt and sampling frequency
dt = float(np.diff(t_vals).mean())
fs = 1.0 / dt
print(f"Sampling frequency: {fs:.1f} Hz")
print(f"Nyquist frequency:  {fs/2:.1f} Hz")

fig, axes = plt.subplots(3, 1, figsize=(10, 12))

# PLotting raw seismogram 
axes[0].plot(t_vals, y_vals, lw=0.8, color="magenta")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("vy (m/s)")
axes[0].set_title("Seismogram — x=0 m")
axes[0].axhline(0, color="gray", lw=0.5) 

#  Power Spectral Density (Welch method)
# Welch averages many overlapping windows 
# nperseg controls frequency resolution: longer = finer freq bins
nperseg = min(1024, len(y_vals) // 4)

freqs_psd, psd = signal.welch(
    y_vals,
    fs=fs,
    nperseg=nperseg,
    noverlap=nperseg // 2,
    window="hann",           # hann window reduces spectral leakage
    scaling="density",       # units ms^2/HZ
)

axes[1].semilogy(freqs_psd, psd, color="magenta")
axes[1].axvline(10, color="orange", lw=1.2, linestyle="--", label="source f0=10 Hz")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("PSD [(m/s)^2/Hz]")
axes[1].set_title("Power Spectral Density (Welch)")
axes[1].set_xlim(0, fs / 2)
axes[1].legend()
axes[1].grid(True, which="both", alpha=0.3)

# spectrogram 
# nperseg_stft controls the time/frequency resolution tradeoff:
# longer window --< finer frequency resolution, coarser time resolution
#  shorter window --> finer time resolution, coarser frequency resolution
# aim fabput ~3-5 cycles of  lowest frequency of interest per window --> CHANGE THIS DEPENDING ON WHICH FREQUENCY 
nperseg_stft = min(512, len(y_vals) // 8)

freqs_stft, t_stft, Sxx = signal.spectrogram(
    y_vals,
    fs=fs,
    nperseg=nperseg_stft,
    noverlap=nperseg_stft * 3 // 4,   # 75% overlap — smooth time axis
    window="hann",
    scaling="density",
)

#log scale for amplitude — seismic signals span many orders of magnitude
Sxx_log = 10 * np.log10(Sxx + 1e-40)   # dB, small floor to avoid log(0)

im = axes[2].pcolormesh(
    t_stft, freqs_stft, Sxx_log,
    shading="gouraud",
    cmap="vanimo",
)
axes[2].axvline(0,  color="black", lw=0.8, linestyle="--", alpha=0.7, label="t=0 (source fires)")
axes[2].axhline(10, color="cyan",  lw=0.8, linestyle="--", alpha=0.7, label="f0=10 Hz")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Frequency (Hz)")
axes[2].set_title("Spectrogram (STFT) — x=0 m")
axes[2].set_ylim(0, min(100, fs / 2))   # cap at 100 Hz or Nyquist
axes[2].legend(loc="upper right", fontsize=8)
fig.colorbar(im, ax=axes[2], label="Power [dB re (m/s)^2/Hz]")

plt.suptitle("Single trace frequency analysis — receiver at x=0 m", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:

# Wiggle plot form shotgather
sg = vel_sg.isel(c=1)  # component to plot (vx/vy)
rec_dim = [d for d in sg.dims if d != "t"][0]
sg = sg.assign_coords({rec_dim: x_line}).rename({rec_dim: "x"})
sg_tx = sg.transpose("t", "x")  # [time, distance]

t = sg_tx["t"].values
x = sg_tx["x"].values
A = sg_tx.values  # shape: [nt, nx]

# Plot settings
trace_step = 30  # plotting  every Nth receiver 
wiggle_scale = 0.7 * np.median(np.diff(x)) * trace_step

fig, ax = plt.subplots(figsize=(14, 8))

for i in range(0, len(x), trace_step):
    tr = A[:, i].copy()
    x0 = x[i]

 # normalization per trace by its own maximum s that wiggles can be seen 
    tr_max = np.max(np.abs(tr))
    if tr_max < 1e-20:      # skip dead traces
        continue
    tr /= tr_max

    # optional: clip wild amplitudes at early time
    #tr = np.clip(tr, -1, 1)

    xwig = x0 + wiggle_scale * tr

    ax.plot(xwig, t, color="black", lw=0.5)
    ax.fill_betweenx(t, x0, xwig,
                     where=(tr > 0),
                     color="black", alpha=0.7, linewidth=0)

ax.invert_yaxis()
ax.set_xlim(x[0] - wiggle_scale, x[-1] + wiggle_scale)
ax.set_xlabel("Distance along interface (m)")
ax.set_ylabel("Time (s)")
ax.set_title("Wiggle shot gather - y=2.0 m")
plt.tight_layout()
plt.show()
